# Run the training pipeline on Colab (GPU)

This notebook only **launches** the scripted pipeline (`python -m pipeline`); all the logic lives in the
`pipeline/` package, so the same run works on any GPU machine.

1. *Runtime → Change runtime type → GPU (T4)*.
2. Put the repository on Drive (or clone it), and set `REPO` and `DATA` below.
3. Run all cells. About 40–50 minutes on a T4: 5 CV folds, the leaky image-level baseline,
   the magnification-shift test and the final model. Finished folds are saved, so if Colab
   disconnects, run all cells again and it resumes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/drive/MyDrive/breast_cancer_histhopalogy_detector'   # the repository folder
DATA = '/content/drive/MyDrive/DATASET MASTERS THESIS/Breast Cancer/Breast_hist.zip'   # BreakHis folder or zip
WORK = '/content/drive/MyDrive/pipeline_work'        # resumable intermediate files
OUT  = '/content/drive/MyDrive/pipeline_artifacts'   # exported model, same layout as deployment_artifacts/
%cd {REPO}

In [ ]:
# Colab already has a CUDA build of torch/torchvision; this adds the pinned training stack.
%pip install -q -r requirements-train.txt

In [ ]:
# Reuse the published patient folds so results are directly comparable.
!python -m pipeline run --data-dir "{DATA}" --work-dir "{WORK}" --out-dir "{OUT}"     --splits-csv deployment_artifacts/split_assignments.csv

In [ ]:
# Optional: the regularised configuration, on the same folds, for comparison.
# !python -m pipeline run --config configs/regularised.json --data-dir "{DATA}" --work-dir "{WORK}" #     --out-dir "{OUT}_regularised" --splits-csv deployment_artifacts/split_assignments.csv

In [ ]:
import json
m = json.load(open(f'{OUT}/metrics.json'))
for k in ['balanced_accuracy', 'malignant_recall', 'benign_recall', 'roc_auc']:
    v = m['headline']['at_selected_threshold'][k]; print(f"{k:<18} {v['mean']:.3f} ± {v['std']:.3f}")
print(json.dumps(m.get('magnification_shift'), indent=1))